In [3]:

import requests, json, sys
sys.path.insert(0, "/app/skills/common/scripts")
from batch_llm import batch_llm

TOKEN = "skZFMehj3STc5EGpVcQPUP5PQRmE4kWEQps0Zso4Rl5Ri3QUfmKRViMkpQ6lkHXZTrnHn0kuQgj6y6x7b6Y0Uz0z1jXPmYCKXVbAvYeZcSFOD7mk6uTEeE3MRSLTanEaUjtrPVEO6DkRdKAt6MOHv0zU4NgWek5XVMcahI6TvYOzLqORIR9J"
BASE   = "https://nzcwegq7.api.sanity.io/v2024-01-01/data/query/production"
MUTATE = "https://nzcwegq7.api.sanity.io/v2024-01-01/data/mutate/production"
H = {"Authorization": f"Bearer {TOKEN}"}

# Pobierz produkty bez shortDescription (non-P, z techSpec)
q = '''*[_type == "product" && !(name match "P-*") &&
        (shortDescription == null || shortDescription == "") &&
        count(technicalSpec) > 0
       ] | order(_id asc) [0...200] {
         _id, name, brand, "categoryName": category->name,
         technicalSpec[]{ label, value }
       }'''

resp = requests.get(BASE, params={"query": q}, headers=H, timeout=60)
products = resp.json().get("result", [])
print(f"Produktów bez shortDescription (z techSpec): {len(products)}")
print("Przykłady:")
for p in products[:3]:
    print(f"  {p['name'][:50]} | specs:{len(p.get('technicalSpec',[]))}")


Produktów bez shortDescription (z techSpec): 77
Przykłady:
  Zaprawa Hydrofobowa Botament M44 Nc Power Kakao Nr | specs:4
  Masa Szpachl.Extrafinish | specs:3
  Urekor Czerwony Tl 10L Sn | specs:12


In [7]:

# Przygotuj prompty dla każdego produktu
def build_prompt(p):
    specs = p.get("technicalSpec", [])
    spec_lines = "\n".join(f"- {s['label']}: {s['value']}" for s in specs[:15])
    brand = p.get("brand") or ""
    cat = p.get("categoryName") or ""
    return f"""Produkt: {p['name']}
Marka: {brand}
Kategoria: {cat}
Parametry techniczne:
{spec_lines}

Napisz po polsku krótki opis produktu (2-3 zdania, max 200 znaków). 
Opis ma być rzeczowy i informacyjny — bazuj TYLKO na podanych parametrach.
Nie wymyślaj cech których nie ma w parametrach.
Nie używaj wyrażeń marketingowych ani superlatyw.
Zwróć TYLKO sam opis, bez cudzysłowów."""

prompts = [build_prompt(p) for p in products]
print(f"Przygotowano {len(prompts)} promptów")
print("Przykład promptu:")
print(prompts[0][:300])


Przygotowano 77 promptów
Przykład promptu:
Produkt: Zaprawa Hydrofobowa Botament M44 Nc Power Kakao Nr 38 5 Kg
Marka: {'_ref': 'brand-czarny', '_type': 'reference'}
Kategoria: Spoiny specjalistyczne
Parametry techniczne:
- Ilość na palecie: 72
- Rodzaj zaprawy: Zaprawa hydrofobowa
- Zmywalność: tak
- Zastosowanie: wewnętrzne, zewnętrzne

Nap


In [11]:

def get_brand_name(brand):
    if not brand:
        return ""
    if isinstance(brand, dict):
        # Wyciągnij nazwę z _ref: "brand-ceresit" → "Ceresit"
        ref = brand.get("_ref", "")
        return ref.replace("brand-", "").replace("-", " ").title()
    return str(brand)

def build_prompt(p):
    specs = p.get("technicalSpec", [])
    spec_lines = "\n".join(f"- {s['label']}: {s['value']}" for s in specs[:12])
    brand = get_brand_name(p.get("brand"))
    cat = p.get("categoryName") or ""
    name = p["name"]
    return (
        f"Produkt: {name}\n"
        f"Marka: {brand}\nKategoria: {cat}\n"
        f"Parametry:\n{spec_lines}\n\n"
        "Napisz po polsku krótki opis produktu (2-3 zdania, do 180 znaków). "
        "Bazuj WYŁĄCZNIE na podanych parametrach. Nie wymyślaj. "
        "Zwróć sam tekst opisu."
    )

prompts = [build_prompt(p) for p in products]
print("Przykład:", prompts[0][:250])
print(f"\nGeneruję opisy dla {len(prompts)} produktów...")

results = await batch_llm(
    prompts,
    system="Jesteś copywriterem sklepu budowlanego. Piszesz krótkie, rzeczowe opisy produktów po polsku. Bazujesz wyłącznie na podanych danych technicznych."
)

ok = sum(1 for r in results if r)
print(f"Wygenerowano: {ok}/{len(results)}")
print("\nPrzykłady:")
for i, (p, desc) in enumerate(zip(products[:5], results[:5])):
    print(f"\n  [{p['name'][:45]}]\n  → {desc}")


Przykład: Produkt: Zaprawa Hydrofobowa Botament M44 Nc Power Kakao Nr 38 5 Kg
Marka: Czarny
Kategoria: Spoiny specjalistyczne
Parametry:
- Ilość na palecie: 72
- Rodzaj zaprawy: Zaprawa hydrofobowa
- Zmywalność: tak
- Zastosowanie: wewnętrzne, zewnętrzne

Napi

Generuję opisy dla 77 produktów...


Batch complete: 77/77 succeeded
Wygenerowano: 77/77

Przykłady:

  [Zaprawa Hydrofobowa Botament M44 Nc Power Kak]
  → Zaprawa hydrofobowa Botament M44 Nc Power Kakao Nr 38 5 Kg. Do zastosowań wewnętrznych i zewnętrznych. Zmywalna. 72 sztuki na palecie.

  [Masa Szpachl.Extrafinish]
  → Masa szpachlowa Extrafinish z serii extrafinish. Ilość na palecie: 54 sztuki.

  [Urekor Czerwony Tl 10L Sn]
  → Urekor Czerwony Tl 10L Sn to podkład antykorozyjny z serii urekor s. Przeznaczony do stali i metalu, do stosowania wewnętrznego i zewnętrznego w temperaturze od +5℃ do +25℃. Aplikacja wałkiem lub pędzlem, efekt matowy.

  [Zestaw Komin Uniwersal Fi180+45 S 6,66Mb]
  → Komin ceramiczny Hoch Uniwersal Fi180+45 S o wysokości 8.66m. Z przyłączem spalin 45°, odporny na pożar sadzy i kondensat. Zastosowanie wewnętrzne.

  [Pakiet Podstaw. Qpro Pod 14]
  → Pakiet Podstawowy Qpro Pod 14. Kominy ceramiczne. Jedna sztuka na palecie.


In [15]:

mutations = []
for p, desc in zip(products, results):
    if desc and len(desc.strip()) > 10:
        mutations.append({
            "patch": {
                "id": p["_id"],
                "set": {"shortDescription": desc.strip()[:300]}
            }
        })

print(f"Mutacji do wysłania: {len(mutations)}")

# Wyślij w jednym batchu (≤100)
resp = requests.post(
    MUTATE, headers={**H, "Content-Type": "application/json"},
    json={"mutations": mutations}, timeout=60
)
resp.raise_for_status()
ok = len(resp.json().get("results", []))
print(f"✅ Sanity updated: {ok} produktów")


Mutacji do wysłania: 77


✅ Sanity updated: 77 produktów
